In [0]:
# Databricks notebook source
# =========================
# Manual Config
# =========================
SNAPSHOT_QUARTER = "2026Q1"
PROMPT_VERSION   = "v6.2.0"
ENDPOINT         = "databricks-gpt-5-2"
RUN_SCOPE        = "PROD"

TEST_MODE        = True
TEST_MAX_GROUPS  = 20

GROUP_DIMS_TO_RUN = ["country", 'brand_name', "d_type", "year"]   # [] 이면 전체,  -> 테스트돌린후 추가
TARGET_Y_FEATURE  = ""   # 예: "07_06_리모컨_사용성", 전체면 ""

PVAL_MAX           = 0.10
ABS_COEF_THRESHOLD = 0.10
USE_IS_DRIVER      = True

MAX_DRIVERS_PER_KEY_FULL    = 4
MAX_DRIVERS_PER_KEY_COMPACT = 3
MAX_DIFF_STATS_FULL         = 10

MAX_TOKENS          = 1600
MAX_RETRIES         = 6
RETRY_SLEEP_SECONDS = 2.0
BACKOFF_BASE        = 1.8

FAILED_MAX_TOKENS          = sandbox.t_online_voc_analysis.voc_wls_dashboard_insight_summary_v62200
FAILED_MAX_RETRIES         = 8
FAILED_RETRY_SLEEP_SECONDS = 3.0
FAILED_BACKOFF_BASE        = 2.0

MODEL_SUMMARY_TABLE = "sandbox.z_jungryo_lee.voc_wls_model_summary_with_corr"
COEF_SUMMARY_TABLE  = "sandbox.z_jungryo_lee.voc_wls_coef_summary_with_corr"

SUMMARY_TABLE = "sandbox.z_jungryo_lee.voc_wls_dashboard_insight_summary"
DETAIL_TABLE  = "sandbox.z_jungryo_lee.voc_wls_dashboard_insight_detail"
FAILED_TABLE  = "sandbox.z_jungryo_lee.voc_wls_dashboard_insight_failed"

print(f"[CONFIG] quarter={SNAPSHOT_QUARTER}, prompt_version={PROMPT_VERSION}, endpoint={ENDPOINT}, run_scope={RUN_SCOPE}")
print(f"[CONFIG] dims={GROUP_DIMS_TO_RUN}, target_y_feature={TARGET_Y_FEATURE}, test_mode={TEST_MODE} (limit={TEST_MAX_GROUPS})")
print(f"[CONFIG] pval<={PVAL_MAX}, abs(coef)>={ABS_COEF_THRESHOLD}, use_is_driver={USE_IS_DRIVER}")
print(f"[CONFIG] normal retries={MAX_RETRIES}, normal sleep={RETRY_SLEEP_SECONDS}s, failed retries={FAILED_MAX_RETRIES}, failed sleep={FAILED_RETRY_SLEEP_SECONDS}s")

# =========================
# Imports
# =========================
from pyspark.sql import functions as F
from pyspark.sql import Row
from datetime import datetime, UTC
import json
import time
import traceback
import re

# =========================
# Prompt: v6 tone 유지
# =========================
SYSTEM_PROMPT = """
You are a senior product strategy planner for TV products.
Return ONLY strict JSON. No preface, no markdown.

Language and style rules:
- Write all outputs in Korean.
- Every sentence must use a formal business tone ending in '~습니다', '~입니다', or equivalent formal style.
- Keep each sentence concise and intuitive for planners and strategists.
- You may use memorable strategic phrasing if it is clear and grounded in the evidence.
- Do not expose raw variable prefixes like '07_02_' or other code-like prefixes in prose.
- If you mention adjusted R-squared, keep the English term exactly as 'adjusted R-squared'.

Analysis rules:
- Base all interpretations only on the evidence provided in the payload.
- Always compare group_keys within the given group_dim.
- The goal is dashboard-ready planning insight, not analyst commentary.
- Translate evidence into planning meaning:
  1. what value customers center on
  2. what makes each market or segment distinct
  3. what planning emphasis should differ across markets
- For each group_key, do not label it as average unless there is no statistically meaningful positive or negative driver after filtering.
- If a group_key has both a strong positive and a meaningful negative signal, mention both.

Key headline rules:
- `key_core` is a card headline, not a descriptive sentence.
- Do not start `key_core` with phrases like '{group_key} 고객은', '{group_key}에서는', or similar wording.
- Write one short punchy sentence that captures the single most distinctive value of that group_key.
- `key_core` should feel like a compact strategic headline or slogan grounded in the evidence.
- Prefer patterns like:
  - '속도가 곧 성능입니다.'
  - 'UI의 구조적 완성도를 중시합니다.'
  - '연결 경험이 사용성을 좌우합니다.'
  - '디자인이 곧 조작 신뢰감입니다.'
- Keep `key_core` short, memorable, and specific.

Summary comment rules:
- `summary_comment` must begin by explaining the meaning of `key_core` in planner-friendly language.
- The first sentence of `summary_comment` may explicitly mention the group_key, for example:
  - '한국 고객은 리모컨 만족도에서 반응성을 가장 중요하게 생각합니다.'
  - '독일 고객은 UI 구조와 탐색 체계의 완성도를 더 중시하는 패턴을 보입니다.'
- After that first sentence, continue with the comparative explanation and evidence.
- Do not begin `summary_comment` with generic framing such as '동일 지표를 비교할 때', '비교 결과', '제공된 유의 동인 기준에서', or similar phrases.
- The first sentence should interpret the headline idea, and the following sentence(s) should explain the supporting evidence.

Driver rules:
- Do not choose which drivers to include.
- The provided drivers are fixed.
- For each provided driver, write only the comment.

Return ONLY strict JSON following this schema:
{
  "overall_core_message": "전체 비교 한 줄 핵심",
  "planning_summary": "기획/전략 관점 요약 멘트",
  "country_insight_cards": [
    {
      "group_key": "Brazil",
      "key_core": "연결 경험이 사용성을 좌우합니다.",
      "summary_comment": "브라질 고객은 리모컨 만족도에서 음성 인식과 연결 경험을 핵심 가치로 인식합니다. 이 시장에서는 음성 제어와 IoT 연동 계열 가치가 만족도를 설명하는 핵심 축입니다.",
      "key_driver_comments": [
        {
          "x_feature": "Voice Control",
          "comment": "핵심 동인으로 해석됩니다."
        }
      ],
      "special_points": [
        "다른 시장보다 연결성이 더 중요한 차별 포인트입니다."
      ]
    }
  ],
  "strategy_takeaways": [
    "시장별로 리모컨의 핵심 가치 정의를 다르게 가져갈 필요가 있습니다."
  ]
}
""".strip()

def build_user_prompt(group_dim: str, y_feature: str) -> str:
    return f"""
You will receive a JSON payload for one y_feature.

Context:
- group_dim: {group_dim}
- y_feature: {y_feature}

Required output:
- overall_core_message
- planning_summary
- country_insight_cards:
  - group_key
  - key_core
  - summary_comment
  - key_driver_comments
  - special_points
- strategy_takeaways

Important writing constraint:
- Every Korean sentence must be in formal business tone such as '~습니다' or '~입니다'.
- Make the writing intuitive for product planners and strategists.
- Do not expose raw prefixes like '07_02_' in output text.
- Focus on planning meaning, not analyst-style metric narration.
""".strip()

# =========================
# Helpers
# =========================
def sql_escape(s):
    return str(s).replace("'", "''")

def rows_to_pylist(lst):
    if not lst:
        return []
    return [dict(x.asDict()) if hasattr(x, "asDict") else dict(x) for x in lst]

def json_dumps_safe(obj):
    return json.dumps(obj, ensure_ascii=False)

def safe_float(v, default=None):
    try:
        if v is None:
            return default
        return float(v)
    except Exception:
        return default

def safe_int(v, default=None):
    try:
        if v is None:
            return default
        return int(v)
    except Exception:
        return default

def clean_feature_name(name: str) -> str:
    if not name:
        return ""
    s = str(name)
    s = re.sub(r"^[0-9]+([_./-][0-9]+)*[_./-]*", "", s)
    s = re.sub(r"^[^A-Za-z가-힣]+", "", s)
    s = s.replace("_", " ").strip()
    return s

def to_direction(coef):
    return "positive" if safe_float(coef, 0.0) >= 0 else "negative"

def sort_drivers(drivers):
    if not drivers:
        return []
    return sorted(drivers, key=lambda d: abs(float(d.get("coef", 0.0) or 0.0)), reverse=True)

def cleanse_driver_names(drivers: list) -> list:
    out = []
    for d in drivers or []:
        x = dict(d)
        x["x_feature"] = clean_feature_name(x.get("x_feature"))
        x["direction"] = x.get("direction") or to_direction(x.get("coef"))
        out.append(x)
    return out

def cleanse_comp_stats(comp_stats: list) -> list:
    out = []
    for item in comp_stats or []:
        x = dict(item)
        x["x_feature"] = clean_feature_name(x.get("x_feature"))
        out.append(x)
    return out

def compute_confidence_level(model_stats: list) -> dict:
    if not model_stats:
        return {
            "level": "low",
            "score_basis": {
                "min_adj_r_squared": None,
                "max_model_p_value": None,
                "min_y_obs": None
            }
        }

    adj_r2_list = [safe_float(x.get("adj_r_squared")) for x in model_stats if safe_float(x.get("adj_r_squared")) is not None]
    pval_list   = [safe_float(x.get("model_p_value")) for x in model_stats if safe_float(x.get("model_p_value")) is not None]
    yobs_list   = [safe_int(x.get("y_obs")) for x in model_stats if safe_int(x.get("y_obs")) is not None]

    min_adj_r2 = min(adj_r2_list) if adj_r2_list else None
    max_pval   = max(pval_list) if pval_list else None
    min_y_obs  = min(yobs_list) if yobs_list else None

    if min_adj_r2 is None or max_pval is None or min_y_obs is None:
        level = "low"
    elif min_adj_r2 >= 0.60 and max_pval <= 0.05 and min_y_obs >= 100:
        level = "high"
    elif min_adj_r2 >= 0.30 and max_pval <= 0.10 and min_y_obs >= 60:
        level = "mid"
    else:
        level = "low"

    return {
        "level": level,
        "score_basis": {
            "min_adj_r_squared": min_adj_r2,
            "max_model_p_value": max_pval,
            "min_y_obs": min_y_obs
        }
    }

def confidence_band_from_level(level: str) -> str:
    if level == "high":
        return "Strong"
    if level == "mid":
        return "Moderate"
    return "Caution"

# =========================
# AI Query
# =========================
def call_ai_query(endpoint: str, request_text: str, temperature: float, max_tokens: int):
    df = spark.createDataFrame([(request_text,)], ["p"])
    out = (
        df.select(
            F.expr(f"""
                ai_query(
                    endpoint => '{endpoint}',
                    request  => p,
                    modelParameters => named_struct(
                        'temperature', {float(temperature)},
                        'max_tokens',  {int(max_tokens)}
                    )
                )
            """).alias("json_out")
        ).first()
    )
    return out["json_out"] if out else None

def call_ai_query_with_retry(endpoint, system_prompt, user_prompt, payload_full_json, payload_compact_json,
                             max_tokens, max_retries, retry_sleep_seconds, backoff_base):
    last_err = None

    for attempt in range(1, max_retries + 1):
        try:
            payload_json = payload_full_json if attempt <= max(2, max_retries // 2) else payload_compact_json
            req = f"[SYSTEM]\n{system_prompt}\n\n[USER]\n{user_prompt}\n\n[PAYLOAD]\n{payload_json}"
            raw = call_ai_query(endpoint, req, temperature=0.0, max_tokens=max_tokens)

            if not raw:
                raise RuntimeError("Empty response from ai_query")

            clean = raw.replace("```json", "").replace("```", "").strip()
            data = json.loads(clean)

            time.sleep(retry_sleep_seconds)
            return data, clean

        except Exception as e:
            wait = retry_sleep_seconds + (backoff_base ** (attempt - 1))
            print(f"[WARN] ai_query failed (attempt {attempt}/{max_retries}): {repr(e)} -> wait {wait:.1f}s")
            time.sleep(wait)
            last_err = e

    raise RuntimeError(f"ai_query failed after {max_retries} retries: {repr(last_err)}")

# =========================
# Load Source Tables
# =========================
model_summary_sdf_org = spark.table(MODEL_SUMMARY_TABLE)
coef_summary_sdf_org  = spark.table(COEF_SUMMARY_TABLE)

if GROUP_DIMS_TO_RUN:
    model_summary_sdf = model_summary_sdf_org.filter(F.col("group_dim").isin(GROUP_DIMS_TO_RUN))
    coef_summary_sdf  = coef_summary_sdf_org.filter(F.col("group_dim").isin(GROUP_DIMS_TO_RUN))
else:
    model_summary_sdf = model_summary_sdf_org
    coef_summary_sdf  = coef_summary_sdf_org

if TARGET_Y_FEATURE:
    model_summary_sdf = model_summary_sdf.filter(F.col("y_feature") == TARGET_Y_FEATURE)
    coef_summary_sdf  = coef_summary_sdf.filter(F.col("y_feature") == TARGET_Y_FEATURE)

print("[INFO] model_summary rows:", model_summary_sdf.count())
print("[INFO] coef_summary rows :", coef_summary_sdf.count())

# =========================
# Driver Filter: v6.1 rule 유지
# =========================
drivers_sdf = coef_summary_sdf.withColumn("abs_coef", F.abs(F.col("coef")))

if USE_IS_DRIVER and "is_driver" in drivers_sdf.columns:
    drivers_sdf = drivers_sdf.filter(F.col("is_driver") == 1)

drivers_sdf = (
    drivers_sdf
    .filter(
        (F.col("p_value") <= F.lit(PVAL_MAX)) &
        (F.col("abs_coef") >= F.lit(ABS_COEF_THRESHOLD))
    )
    .cache()
)

print("[INFO] filtered driver rows:", drivers_sdf.count())

# =========================
# Aggregate
# =========================
ms_agg_sdf = (
    model_summary_sdf
    .groupBy("group_dim", "y_feature")
    .agg(
        F.collect_list(
            F.struct(
                F.col("group_key"),
                F.col("r_squared"),
                F.col("adj_r_squared"),
                F.col("f_statistic"),
                F.col("prob_f").alias("model_p_value"),
                F.col("log_likelihood"),
                F.col("aic"),
                F.col("bic"),
                F.col("y_obs"),
                F.col("cond_no")
            )
        ).alias("model_stats")
    )
)

drivers_overall_agg_sdf = (
    drivers_sdf
    .select(
        "group_dim", "y_feature", "group_key", "x_feature",
        "coef", "p_value", "t_value", "x_obs", "abs_coef"
    )
    .groupBy("group_dim", "y_feature")
    .agg(
        F.collect_list(
            F.struct(
                "group_key",
                "x_feature",
                "coef",
                "p_value",
                "t_value",
                "x_obs",
                "abs_coef"
            )
        ).alias("drivers_filtered")
    )
)

drivers_by_key_agg_sdf = (
    drivers_sdf
    .withColumn("direction", F.when(F.col("coef") >= 0, F.lit("positive")).otherwise(F.lit("negative")))
    .select(
        "group_dim", "y_feature", "group_key",
        "x_feature", "coef", "p_value", "t_value", "x_obs", "abs_coef", "direction"
    )
    .groupBy("group_dim", "y_feature", "group_key")
    .agg(
        F.collect_list(
            F.struct(
                "x_feature",
                "coef",
                "p_value",
                "t_value",
                "x_obs",
                "abs_coef",
                "direction"
            )
        ).alias("key_driver_candidates")
    )
)

payload_base_sdf = (
    ms_agg_sdf
    .join(drivers_overall_agg_sdf, ["group_dim", "y_feature"], "left")
    .cache()
)

total_groups = payload_base_sdf.count()
print("[INFO] payload base groups:", total_groups)

if TEST_MODE and total_groups > TEST_MAX_GROUPS:
    payload_base_sdf = payload_base_sdf.limit(TEST_MAX_GROUPS).cache()
    print(f"[INFO] TEST_MODE -> limited to {TEST_MAX_GROUPS}")

# =========================
# Comparative stats
# =========================
coef_for_compare = (
    coef_summary_sdf
    .select("group_dim", "group_key", "y_feature", "x_feature", "coef")
    .cache()
)

spread_stats_sdf = (
    coef_for_compare
    .groupBy("group_dim", "y_feature", "x_feature")
    .agg(
        F.min("coef").alias("coef_min"),
        F.max("coef").alias("coef_max"),
        F.avg("coef").alias("coef_mean"),
        F.stddev("coef").alias("coef_std")
    )
    .withColumn("coef_spread", F.col("coef_max") - F.col("coef_min"))
)

by_key_list_sdf = (
    coef_for_compare
    .groupBy("group_dim", "y_feature", "x_feature")
    .agg(
        F.collect_list(
            F.struct(F.col("group_key"), F.col("coef"))
        ).alias("by_key_coef_list")
    )
)

comparative_agg_sdf = (
    spread_stats_sdf
    .join(by_key_list_sdf, ["group_dim", "y_feature", "x_feature"], "left")
    .groupBy("group_dim", "y_feature")
    .agg(
        F.collect_list(
            F.struct(
                F.col("x_feature"),
                F.col("coef_min"),
                F.col("coef_max"),
                F.col("coef_mean"),
                F.col("coef_std"),
                F.col("coef_spread"),
                F.col("by_key_coef_list")
            )
        ).alias("x_feature_diff_stats")
    )
)

comp_rows = {(r["group_dim"], r["y_feature"]): r for r in comparative_agg_sdf.collect()}

key_driver_rows = drivers_by_key_agg_sdf.collect()
key_driver_map = {
    (r["group_dim"], r["y_feature"], r["group_key"]): cleanse_driver_names(
        sort_drivers(rows_to_pylist(r["key_driver_candidates"]))
    )
    for r in key_driver_rows
}

# =========================
# Tables
# =========================
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {SUMMARY_TABLE} (
snapshot_quarter STRING,
prompt_version STRING,
group_dim STRING,
y_feature STRING,
y_feature_clean STRING,
overall_core_message STRING,
planning_summary STRING,
strategy_takeaways_json STRING,
confidence_level STRING,
confidence_band STRING,
raw_llm_json STRING,
payload_json STRING,
generated_at TIMESTAMP
) USING delta
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {DETAIL_TABLE} (
snapshot_quarter STRING,
prompt_version STRING,
group_dim STRING,
group_key STRING,
y_feature STRING,
y_feature_clean STRING,
key_core STRING,
summary_comment STRING,
key_drivers_json STRING,
special_points_json STRING,
raw_card_json STRING,
generated_at TIMESTAMP
) USING delta
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {FAILED_TABLE} (
snapshot_quarter STRING,
prompt_version STRING,
group_dim STRING,
y_feature STRING,
error STRING,
payload_json STRING,
failed_at TIMESTAMP
) USING delta
""")

# =========================
# Payload / Mapping
# =========================
def build_payload(base_row, comp_rows_dict, key_driver_map, compact=False):
    group_dim = base_row["group_dim"]
    y_feature = base_row["y_feature"]

    model_summary = rows_to_pylist(base_row["model_stats"])
    confidence_meta = compute_confidence_level(model_summary)

    drivers = rows_to_pylist(base_row["drivers_filtered"]) if base_row["drivers_filtered"] else []
    drivers = cleanse_driver_names(sort_drivers(drivers))
    drivers = drivers[:MAX_DRIVERS_PER_KEY_COMPACT] if compact else drivers[:MAX_DRIVERS_PER_KEY_FULL]

    group_keys = sorted(list({x.get("group_key") for x in model_summary if x.get("group_key") is not None}))

    key_driver_candidates = []
    for group_key in group_keys:
        candidates = key_driver_map.get((group_dim, y_feature, group_key), [])
        candidates = candidates[:MAX_DRIVERS_PER_KEY_COMPACT] if compact else candidates[:MAX_DRIVERS_PER_KEY_FULL]
        key_driver_candidates.append({
            "group_key": group_key,
            "drivers": [
                {
                    "x_feature": d.get("x_feature"),
                    "coef": d.get("coef"),
                    "p_value": d.get("p_value"),
                    "direction": d.get("direction")
                }
                for d in candidates
            ]
        })

    payload = {
        "meta": {
            "snapshot_quarter": SNAPSHOT_QUARTER,
            "prompt_version": PROMPT_VERSION,
            "endpoint": ENDPOINT,
            "run_scope": RUN_SCOPE,
            "generated_at": datetime.now(UTC).strftime("%Y-%m-%dT%H:%M:%SZ"),
            "driver_filter": {
                "p_value_max": PVAL_MAX,
                "abs_coef_threshold": ABS_COEF_THRESHOLD,
                "use_is_driver": USE_IS_DRIVER
            },
            "confidence_rule_basis": confidence_meta["score_basis"]
        },
        "context": {
            "group_dim": group_dim,
            "y_feature_raw": y_feature,
            "y_feature": clean_feature_name(y_feature)
        },
        "model_summary": model_summary,
        "drivers_filtered": drivers,
        "key_driver_candidates": key_driver_candidates
    }

    if not compact:
        comp = comp_rows_dict.get((group_dim, y_feature))
        comp_stats = cleanse_comp_stats(rows_to_pylist(comp["x_feature_diff_stats"])) if comp else []
        comp_stats = sorted(comp_stats, key=lambda x: abs(float(x.get("coef_spread", 0.0) or 0.0)), reverse=True)[:MAX_DIFF_STATS_FULL]
        payload["x_feature_diff_stats"] = comp_stats

    return payload

def llm_json_to_summary_record(group_dim, y_feature, llm_json, payload_str, computed_confidence):
    return {
        "snapshot_quarter": SNAPSHOT_QUARTER,
        "prompt_version": PROMPT_VERSION,
        "group_dim": group_dim,
        "y_feature": y_feature,
        "y_feature_clean": clean_feature_name(y_feature),
        "overall_core_message": llm_json.get("overall_core_message", ""),
        "planning_summary": llm_json.get("planning_summary", ""),
        "strategy_takeaways_json": json_dumps_safe(llm_json.get("strategy_takeaways", [])),
        "confidence_level": computed_confidence.get("level", "low"),
        "confidence_band": confidence_band_from_level(computed_confidence.get("level", "low")),
        "raw_llm_json": json_dumps_safe(llm_json),
        "payload_json": payload_str,
        "generated_at": datetime.now(UTC).strftime("%Y-%m-%d %H:%M:%S")
    }

def llm_json_to_detail_records(group_dim, y_feature, llm_json, key_driver_map):
    detail_records = []
    cards = llm_json.get("country_insight_cards", []) or []

    for card in cards:
        group_key = card.get("group_key", "")
        fixed_drivers = key_driver_map.get((group_dim, y_feature, group_key), [])
        llm_comments = {
            x.get("x_feature"): x.get("comment", "")
            for x in (card.get("key_driver_comments", []) or [])
            if x.get("x_feature")
        }

        key_drivers_final = []
        for d in fixed_drivers:
            key_drivers_final.append({
                "x_feature": d.get("x_feature"),
                "coef": d.get("coef"),
                "p_value": d.get("p_value"),
                "direction": d.get("direction"),
                "comment": llm_comments.get(d.get("x_feature"), "")
            })

        detail_records.append({
            "snapshot_quarter": SNAPSHOT_QUARTER,
            "prompt_version": PROMPT_VERSION,
            "group_dim": group_dim,
            "group_key": group_key,
            "y_feature": y_feature,
            "y_feature_clean": clean_feature_name(y_feature),
            "key_core": card.get("key_core", ""),
            "summary_comment": card.get("summary_comment", ""),
            "key_drivers_json": json_dumps_safe(key_drivers_final),
            "special_points_json": json_dumps_safe(card.get("special_points", [])),
            "raw_card_json": json_dumps_safe(card),
            "generated_at": datetime.now(UTC).strftime("%Y-%m-%d %H:%M:%S")
        })

    return detail_records

# =========================
# Main generate
# =========================
base_rows = payload_base_sdf.collect()
print(f"[INFO] generation targets: {len(base_rows)}")

summary_results = []
detail_results  = []
failed          = []

for i, r in enumerate(base_rows, 1):
    group_dim = r["group_dim"]
    y_feature = r["y_feature"]

    try:
        payload_full = build_payload(r, comp_rows, key_driver_map, compact=False)
        payload_compact = build_payload(r, comp_rows, key_driver_map, compact=True)

        payload_full_str = json.dumps(payload_full, ensure_ascii=False)
        payload_compact_str = json.dumps(payload_compact, ensure_ascii=False)

        computed_confidence = compute_confidence_level(payload_full["model_summary"])

        user_prompt = build_user_prompt(
            group_dim=group_dim,
            y_feature=clean_feature_name(y_feature)
        )

        llm_json, raw = call_ai_query_with_retry(
            endpoint=ENDPOINT,
            system_prompt=SYSTEM_PROMPT,
            user_prompt=user_prompt,
            payload_full_json=payload_full_str,
            payload_compact_json=payload_compact_str,
            max_tokens=MAX_TOKENS,
            max_retries=MAX_RETRIES,
            retry_sleep_seconds=RETRY_SLEEP_SECONDS,
            backoff_base=BACKOFF_BASE
        )

        summary_results.append(
            llm_json_to_summary_record(
                group_dim=group_dim,
                y_feature=y_feature,
                llm_json=llm_json,
                payload_str=payload_compact_str,
                computed_confidence=computed_confidence
            )
        )

        detail_results.extend(
            llm_json_to_detail_records(
                group_dim=group_dim,
                y_feature=y_feature,
                llm_json=llm_json,
                key_driver_map=key_driver_map
            )
        )

        if i % 5 == 0 or i == len(base_rows):
            print(f"[INFO] progress {i}/{len(base_rows)}")

    except Exception as e:
        print(f"[ERROR] ({group_dim}, {y_feature}) failed: {repr(e)}")
        traceback.print_exc()

        failed.append({
            "snapshot_quarter": SNAPSHOT_QUARTER,
            "prompt_version": PROMPT_VERSION,
            "group_dim": group_dim,
            "y_feature": y_feature,
            "error": repr(e),
            "payload_json": json.dumps(
                {"context": {"group_dim": group_dim, "y_feature": y_feature}},
                ensure_ascii=False
            ),
            "failed_at": datetime.now(UTC)
        })

print(f"[INFO] summary success={len(summary_results)}, detail success={len(detail_results)}, failed={len(failed)}")

# =========================
# Save local
# =========================
with open("results_summary_current.json", "w", encoding="utf-8") as f:
    json.dump(summary_results, f, ensure_ascii=False, indent=2)

with open("results_detail_current.json", "w", encoding="utf-8") as f:
    json.dump(detail_results, f, ensure_ascii=False, indent=2)

# =========================
# Merge summary
# =========================
if summary_results:
    summary_sdf = spark.createDataFrame([Row(**rec) for rec in summary_results]).select(
        F.col("snapshot_quarter").cast("string").alias("snapshot_quarter"),
        F.col("prompt_version").cast("string").alias("prompt_version"),
        F.col("group_dim").cast("string").alias("group_dim"),
        F.col("y_feature").cast("string").alias("y_feature"),
        F.col("y_feature_clean").cast("string").alias("y_feature_clean"),
        F.col("overall_core_message").cast("string").alias("overall_core_message"),
        F.col("planning_summary").cast("string").alias("planning_summary"),
        F.col("strategy_takeaways_json").cast("string").alias("strategy_takeaways_json"),
        F.col("confidence_level").cast("string").alias("confidence_level"),
        F.col("confidence_band").cast("string").alias("confidence_band"),
        F.col("raw_llm_json").cast("string").alias("raw_llm_json"),
        F.col("payload_json").cast("string").alias("payload_json"),
        F.to_timestamp("generated_at").alias("generated_at")
    )

    summary_sdf.createOrReplaceTempView("insight_summary_updates_current")

    spark.sql(f"""
        MERGE INTO {SUMMARY_TABLE} AS t
        USING insight_summary_updates_current AS s
        ON  t.snapshot_quarter = s.snapshot_quarter
        AND t.prompt_version   = s.prompt_version
        AND t.group_dim        = s.group_dim
        AND t.y_feature        = s.y_feature
        WHEN MATCHED THEN UPDATE SET
          t.y_feature_clean         = s.y_feature_clean,
          t.overall_core_message    = s.overall_core_message,
          t.planning_summary        = s.planning_summary,
          t.strategy_takeaways_json = s.strategy_takeaways_json,
          t.confidence_level        = s.confidence_level,
          t.confidence_band         = s.confidence_band,
          t.raw_llm_json            = s.raw_llm_json,
          t.payload_json            = s.payload_json,
          t.generated_at            = s.generated_at
        WHEN NOT MATCHED THEN INSERT (
          snapshot_quarter,
          prompt_version,
          group_dim,
          y_feature,
          y_feature_clean,
          overall_core_message,
          planning_summary,
          strategy_takeaways_json,
          confidence_level,
          confidence_band,
          raw_llm_json,
          payload_json,
          generated_at
        )
        VALUES (
          s.snapshot_quarter,
          s.prompt_version,
          s.group_dim,
          s.y_feature,
          s.y_feature_clean,
          s.overall_core_message,
          s.planning_summary,
          s.strategy_takeaways_json,
          s.confidence_level,
          s.confidence_band,
          s.raw_llm_json,
          s.payload_json,
          s.generated_at
        )
    """)
    print(f"[INFO] merged summary rows: {summary_sdf.count()}")

# =========================
# Merge detail
# =========================
if detail_results:
    detail_sdf = spark.createDataFrame([Row(**rec) for rec in detail_results]).select(
        F.col("snapshot_quarter").cast("string").alias("snapshot_quarter"),
        F.col("prompt_version").cast("string").alias("prompt_version"),
        F.col("group_dim").cast("string").alias("group_dim"),
        F.col("group_key").cast("string").alias("group_key"),
        F.col("y_feature").cast("string").alias("y_feature"),
        F.col("y_feature_clean").cast("string").alias("y_feature_clean"),
        F.col("key_core").cast("string").alias("key_core"),
        F.col("summary_comment").cast("string").alias("summary_comment"),
        F.col("key_drivers_json").cast("string").alias("key_drivers_json"),
        F.col("special_points_json").cast("string").alias("special_points_json"),
        F.col("raw_card_json").cast("string").alias("raw_card_json"),
        F.to_timestamp("generated_at").alias("generated_at")
    )

    detail_sdf.createOrReplaceTempView("insight_detail_updates_current")

    spark.sql(f"""
        MERGE INTO {DETAIL_TABLE} AS t
        USING insight_detail_updates_current AS s
        ON  t.snapshot_quarter = s.snapshot_quarter
        AND t.prompt_version   = s.prompt_version
        AND t.group_dim        = s.group_dim
        AND t.group_key        = s.group_key
        AND t.y_feature        = s.y_feature
        WHEN MATCHED THEN UPDATE SET
          t.y_feature_clean     = s.y_feature_clean,
          t.key_core        = s.key_core,
          t.summary_comment     = s.summary_comment,
          t.key_drivers_json    = s.key_drivers_json,
          t.special_points_json = s.special_points_json,
          t.raw_card_json       = s.raw_card_json,
          t.generated_at        = s.generated_at
        WHEN NOT MATCHED THEN INSERT (
          snapshot_quarter,
          prompt_version,
          group_dim,
          group_key,
          y_feature,
          y_feature_clean,
          key_core,
          summary_comment,
          key_drivers_json,
          special_points_json,
          raw_card_json,
          generated_at
        )
        VALUES (
          s.snapshot_quarter,
          s.prompt_version,
          s.group_dim,
          s.group_key,
          s.y_feature,
          s.y_feature_clean,
          s.key_core,
          s.summary_comment,
          s.key_drivers_json,
          s.special_points_json,
          s.raw_card_json,
          s.generated_at
        )
    """)
    print(f"[INFO] merged detail rows: {detail_sdf.count()}")

# =========================
# Save initial failed
# =========================
if failed:
    failed_sdf = spark.createDataFrame([Row(**rec) for rec in failed]).select(
        F.col("snapshot_quarter").cast("string").alias("snapshot_quarter"),
        F.col("prompt_version").cast("string").alias("prompt_version"),
        F.col("group_dim").cast("string").alias("group_dim"),
        F.col("y_feature").cast("string").alias("y_feature"),
        F.col("error").cast("string").alias("error"),
        F.col("payload_json").cast("string").alias("payload_json"),
        F.col("failed_at").cast("timestamp").alias("failed_at")
    )
    failed_sdf.write.mode("append").saveAsTable(FAILED_TABLE)
    print(f"[WARN] initial failed logged: {failed_sdf.count()}")

# =========================
# Failed retry: 더 긴 토큰, 더 긴 대기, 더 많은 retry
# =========================
retry_failed_sdf = (
    spark.table(FAILED_TABLE)
    .filter(F.col("snapshot_quarter") == SNAPSHOT_QUARTER)
    .filter(F.col("prompt_version") == PROMPT_VERSION)
    .select("group_dim", "y_feature")
    .dropDuplicates()
)

retry_keys = {(r["group_dim"], r["y_feature"]) for r in retry_failed_sdf.collect()}
base_rows_retry = [r for r in base_rows if (r["group_dim"], r["y_feature"]) in retry_keys]

print(f"[INFO] failed retry targets: {len(base_rows_retry)}")

retry_summary_results = []
retry_detail_results = []
retry_failed = []

for i, r in enumerate(base_rows_retry, 1):
    group_dim = r["group_dim"]
    y_feature = r["y_feature"]

    try:
        payload_full = build_payload(r, comp_rows, key_driver_map, compact=False)
        payload_compact = build_payload(r, comp_rows, key_driver_map, compact=True)

        payload_full_str = json.dumps(payload_full, ensure_ascii=False)
        payload_compact_str = json.dumps(payload_compact, ensure_ascii=False)

        computed_confidence = compute_confidence_level(payload_full["model_summary"])

        user_prompt = build_user_prompt(
            group_dim=group_dim,
            y_feature=clean_feature_name(y_feature)
        )

        llm_json, raw = call_ai_query_with_retry(
            endpoint=ENDPOINT,
            system_prompt=SYSTEM_PROMPT,
            user_prompt=user_prompt,
            payload_full_json=payload_full_str,
            payload_compact_json=payload_compact_str,
            max_tokens=FAILED_MAX_TOKENS,
            max_retries=FAILED_MAX_RETRIES,
            retry_sleep_seconds=FAILED_RETRY_SLEEP_SECONDS,
            backoff_base=FAILED_BACKOFF_BASE
        )

        retry_summary_results.append(
            llm_json_to_summary_record(
                group_dim=group_dim,
                y_feature=y_feature,
                llm_json=llm_json,
                payload_str=payload_compact_str,
                computed_confidence=computed_confidence
            )
        )

        retry_detail_results.extend(
            llm_json_to_detail_records(
                group_dim=group_dim,
                y_feature=y_feature,
                llm_json=llm_json,
                key_driver_map=key_driver_map
            )
        )

        if i % 5 == 0 or i == len(base_rows_retry):
            print(f"[INFO] failed retry progress {i}/{len(base_rows_retry)}")

    except Exception as e:
        print(f"[ERROR] failed retry ({group_dim}, {y_feature}) failed: {repr(e)}")
        retry_failed.append({
            "snapshot_quarter": SNAPSHOT_QUARTER,
            "prompt_version": PROMPT_VERSION,
            "group_dim": group_dim,
            "y_feature": y_feature,
            "error": repr(e),
            "payload_json": json.dumps(
                {"context": {"group_dim": group_dim, "y_feature": y_feature}},
                ensure_ascii=False
            ),
            "failed_at": datetime.now(UTC)
        })

# retry merge
if retry_summary_results:
    retry_summary_sdf = spark.createDataFrame([Row(**rec) for rec in retry_summary_results]).select(
        F.col("snapshot_quarter").cast("string").alias("snapshot_quarter"),
        F.col("prompt_version").cast("string").alias("prompt_version"),
        F.col("group_dim").cast("string").alias("group_dim"),
        F.col("y_feature").cast("string").alias("y_feature"),
        F.col("y_feature_clean").cast("string").alias("y_feature_clean"),
        F.col("overall_core_message").cast("string").alias("overall_core_message"),
        F.col("planning_summary").cast("string").alias("planning_summary"),
        F.col("strategy_takeaways_json").cast("string").alias("strategy_takeaways_json"),
        F.col("confidence_level").cast("string").alias("confidence_level"),
        F.col("confidence_band").cast("string").alias("confidence_band"),
        F.col("raw_llm_json").cast("string").alias("raw_llm_json"),
        F.col("payload_json").cast("string").alias("payload_json"),
        F.to_timestamp("generated_at").alias("generated_at")
    )
    retry_summary_sdf.createOrReplaceTempView("insight_summary_retry_current")
    spark.sql(f"""
        MERGE INTO {SUMMARY_TABLE} AS t
        USING insight_summary_retry_current AS s
        ON  t.snapshot_quarter = s.snapshot_quarter
        AND t.prompt_version   = s.prompt_version
        AND t.group_dim        = s.group_dim
        AND t.y_feature        = s.y_feature
        WHEN MATCHED THEN UPDATE SET
          t.y_feature_clean         = s.y_feature_clean,
          t.overall_core_message    = s.overall_core_message,
          t.planning_summary        = s.planning_summary,
          t.strategy_takeaways_json = s.strategy_takeaways_json,
          t.confidence_level        = s.confidence_level,
          t.confidence_band         = s.confidence_band,
          t.raw_llm_json            = s.raw_llm_json,
          t.payload_json            = s.payload_json,
          t.generated_at            = s.generated_at
        WHEN NOT MATCHED THEN INSERT (
          snapshot_quarter, prompt_version, group_dim, y_feature, y_feature_clean,
          overall_core_message, planning_summary, strategy_takeaways_json,
          confidence_level, confidence_band, raw_llm_json, payload_json, generated_at
        )
        VALUES (
          s.snapshot_quarter, s.prompt_version, s.group_dim, s.y_feature, s.y_feature_clean,
          s.overall_core_message, s.planning_summary, s.strategy_takeaways_json,
          s.confidence_level, s.confidence_band, s.raw_llm_json, s.payload_json, s.generated_at
        )
    """)

if retry_detail_results:
    retry_detail_sdf = spark.createDataFrame([Row(**rec) for rec in retry_detail_results]).select(
        F.col("snapshot_quarter").cast("string").alias("snapshot_quarter"),
        F.col("prompt_version").cast("string").alias("prompt_version"),
        F.col("group_dim").cast("string").alias("group_dim"),
        F.col("group_key").cast("string").alias("group_key"),
        F.col("y_feature").cast("string").alias("y_feature"),
        F.col("y_feature_clean").cast("string").alias("y_feature_clean"),
        F.col("key_core").cast("string").alias("key_core"),
        F.col("summary_comment").cast("string").alias("summary_comment"),
        F.col("key_drivers_json").cast("string").alias("key_drivers_json"),
        F.col("special_points_json").cast("string").alias("special_points_json"),
        F.col("raw_card_json").cast("string").alias("raw_card_json"),
        F.to_timestamp("generated_at").alias("generated_at")
    )
    retry_detail_sdf.createOrReplaceTempView("insight_detail_retry_current")
    spark.sql(f"""
        MERGE INTO {DETAIL_TABLE} AS t
        USING insight_detail_retry_current AS s
        ON  t.snapshot_quarter = s.snapshot_quarter
        AND t.prompt_version   = s.prompt_version
        AND t.group_dim        = s.group_dim
        AND t.group_key        = s.group_key
        AND t.y_feature        = s.y_feature
        WHEN MATCHED THEN UPDATE SET
          t.y_feature_clean     = s.y_feature_clean,
          t.key_core        = s.key_core,
          t.summary_comment     = s.summary_comment,
          t.key_drivers_json    = s.key_drivers_json,
          t.special_points_json = s.special_points_json,
          t.raw_card_json       = s.raw_card_json,
          t.generated_at        = s.generated_at
        WHEN NOT MATCHED THEN INSERT (
          snapshot_quarter, prompt_version, group_dim, group_key, y_feature, y_feature_clean,
          key_core, summary_comment, key_drivers_json, special_points_json, raw_card_json, generated_at
        )
        VALUES (
          s.snapshot_quarter, s.prompt_version, s.group_dim, s.group_key, s.y_feature, s.y_feature_clean,
          s.key_core, s.summary_comment, s.key_drivers_json, s.special_points_json, s.raw_card_json, s.generated_at
        )
    """)

# retry success remove from failed
successful_retry_keys = [(rec["group_dim"], rec["y_feature"]) for rec in retry_summary_results]
if successful_retry_keys:
    delete_condition = " OR ".join([
        f"(group_dim = '{sql_escape(g)}' AND y_feature = '{sql_escape(y)}')"
        for g, y in successful_retry_keys
    ])
    spark.sql(f"""
        DELETE FROM {FAILED_TABLE}
        WHERE snapshot_quarter = '{sql_escape(SNAPSHOT_QUARTER)}'
          AND prompt_version = '{sql_escape(PROMPT_VERSION)}'
          AND ({delete_condition})
    """)
    print(f"[INFO] removed successful retry keys from failed table: {len(successful_retry_keys)}")

# retry failed append
if retry_failed:
    retry_failed_sdf = spark.createDataFrame([Row(**rec) for rec in retry_failed]).select(
        F.col("snapshot_quarter").cast("string").alias("snapshot_quarter"),
        F.col("prompt_version").cast("string").alias("prompt_version"),
        F.col("group_dim").cast("string").alias("group_dim"),
        F.col("y_feature").cast("string").alias("y_feature"),
        F.col("error").cast("string").alias("error"),
        F.col("payload_json").cast("string").alias("payload_json"),
        F.col("failed_at").cast("timestamp").alias("failed_at")
    )
    retry_failed_sdf.write.mode("append").saveAsTable(FAILED_TABLE)
    print(f"[WARN] failed retry logged: {retry_failed_sdf.count()}")

# dedup failed by latest failed_at
spark.sql(f"""
MERGE INTO {FAILED_TABLE} AS t
USING (
  SELECT
    snapshot_quarter,
    prompt_version,
    group_dim,
    y_feature,
    failed_at
  FROM (
    SELECT
      snapshot_quarter,
      prompt_version,
      group_dim,
      y_feature,
      failed_at,
      ROW_NUMBER() OVER (
        PARTITION BY snapshot_quarter, prompt_version, group_dim, y_feature
        ORDER BY failed_at DESC
      ) AS rn
    FROM {FAILED_TABLE}
    WHERE snapshot_quarter = '{sql_escape(SNAPSHOT_QUARTER)}'
      AND prompt_version = '{sql_escape(PROMPT_VERSION)}'
  ) d
  WHERE rn > 1
) AS s
ON  t.snapshot_quarter = s.snapshot_quarter
AND t.prompt_version   = s.prompt_version
AND t.group_dim        = s.group_dim
AND t.y_feature        = s.y_feature
AND t.failed_at        = s.failed_at
WHEN MATCHED THEN DELETE
""")

print("[INFO] failed table deduplicated")

display(
    spark.table(SUMMARY_TABLE)
        .filter(F.col("snapshot_quarter") == SNAPSHOT_QUARTER)
        .filter(F.col("prompt_version") == PROMPT_VERSION)
        .orderBy(F.desc("generated_at"))
)
